# KLWX multi-cluster tracker: May 27, 2024

This is a second real DC-area event chosen to demonstrate multiple simultaneous storm clusters. NWS Baltimore/Washington documented scattered to numerous thunderstorms, including severe wind reports around the northwestern DC suburbs and Baltimore area, that evening. We use six adjacent KLWX Level II scans from 22:15–22:48 UTC.

Each thresholded cluster starts its own Kalman track. The global nearest-neighbour association matches every active track to at most one later cluster; new clusters create tracks and briefly missing clusters coast before retirement. This is a multi-object baseline—not yet full split/merge lineage analysis.

In [ ]:
%pip install -e "../.[nexrad]"

In [ ]:
from pathlib import Path
import sys
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import matplotlib.pyplot as plt
import numpy as np

repo = Path.cwd().parent
sys.path.insert(0, str(repo / 'src'))
from storm_motion.nexrad import load_lowest_reflectivity_sweep
from storm_motion.tracking import MultiClusterTracker, RadarFrame, cluster_cells, detect_cells

bucket = 'unidata-nexrad-level2'
keys = [
    '2024/05/27/KLWX/KLWX20240527_221509_V06',
    '2024/05/27/KLWX/KLWX20240527_222142_V06',
    '2024/05/27/KLWX/KLWX20240527_222826_V06',
    '2024/05/27/KLWX/KLWX20240527_223511_V06',
    '2024/05/27/KLWX/KLWX20240527_224155_V06',
    '2024/05/27/KLWX/KLWX20240527_224840_V06',
]
data_dir = repo / 'data' / 'klwx_20240527_multicluster'
data_dir.mkdir(parents=True, exist_ok=True)
s3 = boto3.client('s3', region_name='us-east-1', config=Config(signature_version=UNSIGNED))
paths = []
for key in keys:
    path = data_dir / Path(key).name
    if not path.exists():
        s3.download_file(bucket, key, str(path))
    paths.append(path)
frames = [load_lowest_reflectivity_sweep(path) for path in paths]
print(*[frame.observed_at.isoformat() for frame in frames], sep='\n')

In [ ]:
# Keep the DC–Baltimore region and deliberately preserve separate convective cores.
roi_km = dict(xmin=-35, xmax=125, ymin=-45, ymax=120)
def in_roi(frame):
    x_km, y_km = frame.x_m / 1_000, frame.y_m / 1_000
    inside = (x_km >= roi_km['xmin']) & (x_km <= roi_km['xmax']) & (y_km >= roi_km['ymin']) & (y_km <= roi_km['ymax'])
    return RadarFrame(frame.observed_at, np.where(inside, frame.reflectivity_dbz, np.nan), frame.x_m, frame.y_m, frame.radar_latitude_deg, frame.radar_longitude_deg)

def clusters_for(frame):
    # Higher cores first: preserving separated convective cores is useful for multi-object tracking.
    for threshold_dbz, min_gates in [(40, 12), (35, 16), (30, 20)]:
        clusters = cluster_cells(detect_cells(frame, threshold_dbz=threshold_dbz, min_gates=min_gates), linkage_distance_m=10_000)
        if len(clusters) >= 2:
            return clusters, threshold_dbz, min_gates
    # Keep the scan even when a temporary merge leaves just one cluster.
    return clusters, threshold_dbz, min_gates

roi_frames = [in_roi(frame) for frame in frames]
# The source frames are no longer needed after producing the masked detection views.
del frames
cluster_results = [clusters_for(frame) for frame in roi_frames]
for frame, (clusters, threshold_dbz, min_gates) in zip(roi_frames, cluster_results):
    print(f'{frame.observed_at:%H:%M:%S UTC}: {len(clusters)} clusters at {threshold_dbz:.0f} dBZ / {min_gates} gates')

In [ ]:
multi_tracker = MultiClusterTracker(
    measurement_std_m=4_000, acceleration_std_mps2=0.01,
    association_distance_m=30_000, max_missed_scans=1,
)
updates = []
for frame, (clusters, _, _) in zip(roi_frames, cluster_results):
    update = multi_tracker.update(frame.observed_at, clusters)
    updates.append(update)
    observed = [track for track in update.tracks if track.observed_cluster is not None]
    summary = ', '.join(f'T{track.track_id}:{track.status}' for track in update.tracks) or 'no active tracks'
    print(f'{frame.observed_at:%H:%M:%S UTC}: {len(observed)} observed / {len(update.tracks)} active — {summary}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True, sharey=True)
colors = plt.get_cmap('tab20')
for ax, frame, update in zip(axes.flat, roi_frames, updates):
    image = ax.pcolormesh(frame.x_m / 1_000, frame.y_m / 1_000, frame.reflectivity_dbz, cmap='turbo', vmin=0, vmax=65)
    for track in update.tracks:
        color = colors((track.track_id - 1) % 20)
        if track.observed_cluster is not None:
            cluster = track.observed_cluster
            ax.scatter(cluster.centroid_x_m / 1_000, cluster.centroid_y_m / 1_000, s=145, c=[color], edgecolors='white', linewidths=1.4, zorder=5)
            ax.annotate(f'T{track.track_id} · {track.status}', (cluster.centroid_x_m / 1_000, cluster.centroid_y_m / 1_000), xytext=(6, 6), textcoords='offset points', color='white', weight='bold', fontsize=9)
        else:
            ax.scatter(track.estimate.x_m / 1_000, track.estimate.y_m / 1_000, s=130, facecolors='none', edgecolors=[color], linewidths=2, linestyle='--', zorder=5)
            ax.annotate(f'T{track.track_id} · coasting', (track.estimate.x_m / 1_000, track.estimate.y_m / 1_000), xytext=(6, 6), textcoords='offset points', color=color, weight='bold', fontsize=9)
    ax.set(title=f'{frame.observed_at:%H:%M:%S UTC}', xlim=(roi_km['xmin'], roi_km['xmax']), ylim=(roi_km['ymin'], roi_km['ymax']), aspect='equal')
fig.supxlabel('km east of KLWX'); fig.supylabel('km north of KLWX')
fig.suptitle('May 27, 2024: multi-cluster Kalman tracks', fontsize=15, weight='bold')
fig.colorbar(image, ax=axes.ravel().tolist(), label='reflectivity (dBZ)'); plt.show(); plt.close(fig)

## What to inspect

Each colored point is an observed cluster with a persistent track ID. `new` means a cluster first appeared, `matched` means it was associated with the same prior track, and `coasting` means its prior predicted position is retained through a one-scan missed detection. If two cores merge into one thresholded region, the baseline will associate one track and let the other coast/retire; preserving explicit split/merge lineage is the next refinement.